# Atividade 6 — Tabelas 5 e Tabela 1.1.1

Composição da desocupação por sexo (2012 e 2026) e horas semanais de cuidados e afazeres domésticos (2022).

In [ ]:
from pathlib import Path
import pandas as pd

PASTA = Path.cwd()
CHAVES = ["Sigla", "Código", "Estado"]

## 1. Abrir as duas Tabelas 5

In [ ]:
tabela_2012 = pd.read_csv(PASTA / "Tabela5-sem_emprego_2012.csv", sep=";", decimal=",")
tabela_2026 = pd.read_csv(PASTA / "Tabela5-sem_emprego_2026.csv", sep=";", decimal=",")
display(tabela_2012.head())
display(tabela_2026.head())

## 2. Juntar 2012 e 2026

In [ ]:
comparacao = tabela_2012.merge(tabela_2026, on=CHAVES, how="inner", validate="one_to_one")
comparacao.head()

## 3. Abrir e organizar a Tabela 1.1.1 do IBGE

In [ ]:
ibge_2022 = pd.read_excel(PASTA / "Tabela 1.1.1.xls", sheet_name="2022", header=None, skiprows=8, usecols="A:H")
ibge_2022.columns = [
    "Estado", "Horas - total", "Horas - total branca", "Horas - total preta ou parda",
    "Horas - homem branco", "Horas - homem preto ou pardo",
    "Horas - mulher branca", "Horas - mulher preta ou parda",
]
colunas_horas = ibge_2022.columns.drop("Estado")
ibge_2022[colunas_horas] = ibge_2022[colunas_horas].apply(pd.to_numeric, errors="coerce")
ibge_2022 = ibge_2022.dropna(subset=["Estado", "Horas - total"])
ibge_2022.head()

## 4. Cruzar e comparar

In [ ]:
geral = comparacao.merge(ibge_2022, on="Estado", how="inner", validate="one_to_one")
geral["Variação homens (p.p.)"] = (geral["Desocupados - homens (2026 T1)"] - geral["Desocupados - homens (2012 T1)"]).round(1)
geral["Variação mulheres (p.p.)"] = (geral["Desocupados - mulheres (2026 T1)"] - geral["Desocupados - mulheres (2012 T1)"]).round(1)
geral.head()

## 5. Validar e exportar

In [ ]:
assert len(geral) == 27
assert geral["Código"].is_unique
assert not geral.isna().any().any()
geral.to_csv(PASTA / "geral.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
print(f"Resultado: {geral.shape[0]} linhas e {geral.shape[1]} colunas")
geral